# 1. 引入模块


In [5]:
# 从 typing 库导入类型注解工具：List(列表类型)、Dict(字典类型)、Any(任意类型)
from typing import List,Dict,Any
# 从 langchain.agents 模块导入 create_agent 函数，用于创建智能Agent代理
from langchain.agents import create_agent
# 从 langchain.tools 模块导入 tool 装饰器，用来自定义工具函数
from langchain.tools import tool
# 从 langchain.chat_models 模块导入模型初始化函数，用于加载各类大语言模型
from  langchain.chat_models import init_chat_model

```python
主管Agent（Supervisor）
    ｜- 任务分解
    ｜- 拆解分配-----｜
    ｜- 整合结果     |
                    |
                    |
            |-----------------|
        代码助手             研究助手

# 2. 执行智能体-负责执行代码

In [9]:
from typing import List,Dict,Any  # 导入类型注解：列表、字典、任意类型
from langchain.agents import create_agent  # 导入创建Agent的函数
from langchain.tools import tool  # 导入工具装饰器
from langchain.chat_models import init_chat_model  # 导入初始化大模型函数

@tool  # 将函数注册为LangChain可调用工具
def execute_python_code(code:str)-> str:  # 定义函数：接收代码字符串，返回字符串
    """执行python代码并返回结果，
       用于测试代码片段、计算、简单脚本执行 
    Args: 
        code:要执行的python代码字符串"""
    safe_globals = {}  # 定义全局环境字典（用于安全沙箱）
    exec_globals = {}  # 定义局部环境字典（存放执行后的变量）

    try:  # 捕获代码执行异常
        exec(code,safe_globals,exec_globals)  # 执行代码（⚠️ 原代码变量命名不规范：第2个是全局，第3个是局部）

        if "result" in exec_globals:  # 判断是否定义了result变量
            output = str(exec_globals["result"])  # 获取result并转字符串
        else:  # 没有result时
            output = "代码返回成功（无返回值）"  # 给出提示

    except Exception as e:  # 捕获所有错误
        return f"执行错误：{str(e)}"  # 返回错误信息
    
    return output  

In [ ]:
model = init_chat_model(
    model="ollama:qwen3.6:latest",        # 指定模型名称：ollama+模型名
    temperature=0.5,                      # 温度系数：0=稳定，1=创意
    base_url="http://192.168.8.21:11434"  # Ollama 服务地址
)

# 创建代码执行Agent的函数
def create_code_agent():
    """创建代码助手子agent"""              # 函数说明文档
    return create_agent(                # 返回创建好的Agent
        model=model,                    # 绑定上面定义的大模型
        tools=[execute_python_code],    # ⚠️ 修复：末尾加逗号
        system_prompt="""  # 系统提示词：定义Agent角色
你是一个专业的代码助手，擅长编写和执行Python代码。
        
你的职责：
1. 根据用户需求编写Python代码
2. 执行代码并解释结果
3. 调试代码错误

重要提示：
- 使用 execute_python_code 工具执行代码
- 对于复杂问题，先拆解为多个小步骤
- 返回结果时要清晰、易懂
        """
    )

# 3. 搜索智能体-负责网络搜索

In [ ]:
@tool                                          # 注册为Agent可调用网络搜索工具
def web_search(query:str,max_results:int=3)->str: # 接收搜索关键词，限定结果条数，返回文本信息
    """
    模型网络搜索工具，在实际开发时，使用requests,pyscra等框架
    Args：
        query:搜索查询的句子
        max_results:做最大返回结果
    """
    mock_results = {                             # 模拟网络搜索数据库 · 内置缓存词条
        "LangChain": "LangChain 是一个用于构建 LLM 应用的开源框架。最新版本 v1.0 专注于 Agent 开发。",
        "Python": "Python 是广泛使用的编程语言，以简洁易读著称。",
        "AI": "人工智能正在快速发展，LLM 是当前热点方向。"
    }

    for key,value in mock_results.items():        # 遍历预设词条库进行关键词匹配
        if key.lower() in query.lower():          # 忽略大小写模糊匹配，提升检索容错率
            return f"搜索结果：\n-{key}:{value}"    # 匹配成功，格式化返回对应信息
    return f"未找到关于{query}的详细信息,建议使用其他更具体的关键词" # 无匹配词条时默认提示

In [ ]:
def create_search_aagent():
    return create_agent(
        model=model,
        tools=[web_search],
        system_prompt="""
        你是一个专业的研究助手，擅长信息搜索和知识整理。
        
你的职责：
1. 搜索用户需要的信息
2. 整合搜索结果，提供总结
3. 引用信息来源

重要提示：
- 使用 web_search 工具获取信息
- 搜索结果可能不完整，如有需要可进行多次搜索
- 用结构化的方式呈现信息
        """
    )

# 4. 主管智能体-负责分发任务

# 5. 调用